# 02. Data Reliability Assessment

## Overview

Validate the quality and consistency of the raw datasets before downstream analysis.

### Validation Areas

- Data loading
- Schema
- Completeness
- Key integrity
- Business rules
- Cross-table consistency


## 1. Setup and configuration

In [38]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

# Locate project root
project_root = Path.cwd().resolve()

while not (project_root / "data").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("Project root not found.")
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"

print(f"Project root : {project_root}")
print(f"Raw directory: {raw_dir}")

Project root : /home/pamern/Projects/fashion-ecommerce-analytics
Raw directory: /home/pamern/Projects/fashion-ecommerce-analytics/data/raw


In [20]:
PRIMARY_KEYS = {
    "customers": ["customer_id"],
    "products": ["product_id"],
    "geography": ["zip"],
    "orders": ["order_id"],
    "returns": ["return_id"],
    "reviews": ["review_id"],
    "promotions": ["promo_id"],
}

GRAIN_KEYS = {
    "order_items": ["order_id", "product_id"],
    "payments": ["order_id"],
    "shipments": ["order_id"],
    "inventory": ["snapshot_date", "product_id"],
    "web_traffic": ["date", "traffic_source"],
    "sales": ["Date"],
}

FOREIGN_KEYS = [
    ("customers", "zip", "geography", "zip"),
    ("orders", "customer_id", "customers", "customer_id"),
    ("orders", "zip", "geography", "zip"),

    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "promo_id", "promotions", "promo_id"),
    ("order_items", "promo_id_2", "promotions", "promo_id"),

    ("payments", "order_id", "orders", "order_id"),
    ("shipments", "order_id", "orders", "order_id"),

    ("returns", "order_id", "orders", "order_id"),
    ("returns", "product_id", "products", "product_id"),

    ("reviews", "order_id", "orders", "order_id"),
    ("reviews", "customer_id", "customers", "customer_id"),
    ("reviews", "product_id", "products", "product_id"),

    ("inventory", "product_id", "products", "product_id"),
]

DATE_COLUMNS = {
    "customers": ["signup_date"],
    "orders": ["order_date"],
    "shipments": ["ship_date", "delivery_date"],
    "returns": ["return_date"],
    "reviews": ["review_date"],
    "inventory": ["snapshot_date"],
    "promotions": ["start_date", "end_date"],
    "web_traffic": ["date"],
    "sales": ["Date"],
}

## 2. Load and preview data

In [39]:
table_overview = pd.DataFrame(
    [
        {
            "table": table_name,
            "rows": len(df),
            "columns": len(df.columns),
            "dtypes": ", ".join(
                f"{col} ({dtype})"
                for col, dtype in df.dtypes.items()
            ),
        }
        for table_name, df in tables.items()
    ]
).sort_values("table").reset_index(drop=True)

display(table_overview)

,table,rows,columns,dtypes
0,customers,121930,7,"customer_id (int64), zip (int64), city (str), signup_date (str), gender (str), age_group (str), acquisition_channel (str)"
1,geography,39948,4,"zip (int64), city (str), region (str), district (str)"
2,inventory,60247,17,"snapshot_date (str), product_id (int64), stock_on_hand (int64), units_received (int64), units_sold (int64), stockout_days (int64), days_of_supply (float64), fill_rate (float64), stockout_flag (int64), overstock_flag (int64), reorder_flag (int64), sell_through_rate (float64), product_name (str), category (str), segment (str), year (int64), month (int64)"
3,order_items,714669,7,"order_id (int64), product_id (int64), quantity (int64), unit_price (float64), discount_amount (float64), promo_id (str), promo_id_2 (str)"
4,orders,646945,8,"order_id (int64), order_date (str), customer_id (int64), zip (int64), order_status (str), payment_method (str), device_type (str), order_source (str)"
5,payments,646945,4,"order_id (int64), payment_method (str), payment_value (float64), installments (int64)"
6,products,2412,8,"product_id (int64), product_name (str), category (str), segment (str), size (str), color (str), price (float64), cogs (float64)"
7,promotions,50,10,"promo_id (str), promo_name (str), promo_type (str), discount_value (float64), start_date (str), end_date (str), applicable_category (str), promo_channel (str), stackable_flag (int64), min_order_value (int64)"
8,returns,39939,7,"return_id (str), order_id (int64), product_id (int64), return_date (str), return_reason (str), return_quantity (int64), refund_amount (float64)"
9,reviews,113551,7,"review_id (str), order_id (int64), product_id (int64), customer_id (int64), review_date (str), rating (int64), review_title (str)"


In [27]:
for table_name, df in tables.items():
    print(f"\n{table_name} — {df.shape[0]:,} rows × {df.shape[1]} columns")
    display(df.head(3))


customers — 121,930 rows × 7 columns


,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search



geography — 39,948 rows × 4 columns


,zip,city,region,district
0,15201,Hai Phong,East,District #13
1,15202,Phu Ly,East,District #13
2,15203,Viet Tri,East,District #13



inventory — 60,247 rows × 17 columns


...

,Date,Revenue,COGS
0,2023-01-01,2665507.20,2518885.15
1,2023-01-02,1280007.89,1136463.00
2,2023-01-03,1015899.51,822721.12



shipments — 566,067 rows × 4 columns


,order_id,ship_date,delivery_date,shipping_fee
0,1,2012-07-07,2012-07-11,1.37
1,2,2012-07-06,2012-07-10,2.60
2,3,2012-07-04,2012-07-07,2.38



web_traffic — 3,652 rows × 7 columns


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct


## 3. Completeness

In [19]:
completeness = pd.concat(
    [
        pd.DataFrame({
            "table": table_name,
            "column": df.columns,
            "missing_count": df.isna().sum().values,
            "missing_pct": df.isna().mean().mul(100).round(2).values,
        })
        for table_name, df in tables.items()
    ],
    ignore_index=True,
)

display(
    completeness
    .query("missing_count > 0")
    .sort_values(["missing_pct", "table"], ascending=[False, True])
    .reset_index(drop=True)
)

,table,column,missing_count,missing_pct
0,order_items,promo_id_2,714463,99.97
1,promotions,applicable_category,40,80.00
2,order_items,promo_id,438353,61.34


## 4. Key and relationships


In [22]:
pk_results = []

for table_name, keys in PRIMARY_KEYS.items():
    if table_name not in tables:
        continue

    df = tables[table_name]

    duplicate_count = df.duplicated(subset=keys).sum()

    pk_results.append({
        "table": table_name,
        "primary_key": ", ".join(keys),
        "duplicate_rows": duplicate_count,
        "status": "pass" if duplicate_count == 0 else "fail",
    })

pk_check = pd.DataFrame(pk_results)

display(pk_check)

,table,primary_key,duplicate_rows,status
0,customers,customer_id,0,pass
1,products,product_id,0,pass
2,geography,zip,0,pass
3,orders,order_id,0,pass
4,returns,return_id,0,pass
5,reviews,review_id,0,pass
6,promotions,promo_id,0,pass


In [26]:
fk_results = []

for child_table, child_key, parent_table, parent_key in FOREIGN_KEYS:
    if child_table not in tables or parent_table not in tables:
        continue

    child = tables[child_table]
    parent = tables[parent_table]

    orphan_count = (
        child.loc[
            child[child_key].notna(),
            child_key,
        ]
        .isin(parent[parent_key])
        .pipe(lambda x: (~x).sum())
    )

    fk_results.append({
        "relationship": f"{child_table}.{child_key} -> {parent_table}.{parent_key}",
        "orphan_rows": orphan_count,
        "status": "pass" if orphan_count == 0 else "fail",
    })

fk_check = pd.DataFrame(fk_results)

display(fk_check)

,relationship,orphan_rows,status
0,customers.zip -> geography.zip,0,pass
1,orders.customer_id -> customers.customer_id,0,pass
2,orders.zip -> geography.zip,0,pass
3,order_items.order_id -> orders.order_id,0,pass
4,order_items.product_id -> products.product_id,0,pass
5,order_items.promo_id -> promotions.promo_id,0,pass
6,order_items.promo_id_2 -> promotions.promo_id,0,pass
7,payments.order_id -> orders.order_id,0,pass
8,shipments.order_id -> orders.order_id,0,pass
9,returns.order_id -> orders.order_id,0,pass


## 7. Business and temporal validity


In [30]:
temporal_check = pd.DataFrame([
    {
        "rule": "ship_date >= order_date",
        "violations": (
            tables["shipments"]
            .merge(tables["orders"][["order_id", "order_date"]], on="order_id")
            .eval("ship_date < order_date")
            .sum()
        ),
    },
    {
        "rule": "delivery_date >= ship_date",
        "violations": (
            tables["shipments"]
            .eval("delivery_date < ship_date")
            .sum()
        ),
    },
    {
        "rule": "end_date >= start_date",
        "violations": (
            tables["promotions"]
            .eval("end_date < start_date")
            .sum()
        ),
    },
])

temporal_check["status"] = np.where(temporal_check["violations"] == 0, "pass", "fail")

display(temporal_check)

,rule,violations,status
0,ship_date >= order_date,0,pass
1,delivery_date >= ship_date,0,pass
2,end_date >= start_date,0,pass


In [41]:
NUMERIC_RULES = [
    ("order_items", "quantity", lambda x: x > 0),
    ("order_items", "unit_price", lambda x: x >= 0),
    ("payments", "payment_value", lambda x: x >= 0),
    ("reviews", "rating", lambda x: x.between(1, 5)),
]

numeric_check = pd.DataFrame([
    {
        "table": table,
        "column": column,
        "violations": (~rule(tables[table][column].dropna())).sum(),
        "status": "pass" if (~rule(tables[table][column].dropna())).sum() == 0 else "fail",
    }
    for table, column, rule in NUMERIC_RULES
    if table in tables and column in tables[table]
])

display(numeric_check)

,table,column,violations,status
0,order_items,quantity,0,pass
1,order_items,unit_price,0,pass
2,payments,payment_value,0,pass
3,reviews,rating,0,pass


In [43]:
for table, column in [
    ("customers", "gender"),
    ("orders", "order_status"),
    ("payments", "payment_method"),
]:
    print(f"\n{table}.{column}")
    display(
        tables[table][column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )


customers.gender


,gender,count
0,Female,59640
1,Male,57457
2,Non-binary,4833



orders.order_status


,order_status,count
0,delivered,516716
1,cancelled,59462
2,returned,36142
3,shipped,13773
4,paid,13577
5,created,7275



payments.payment_method


,payment_method,count
0,credit_card,356352
1,paypal,97018
2,cod,96681
3,apple_pay,64763
4,bank_transfer,32131


# Summary
The dataset schema is generally consistent, with appropriate data types for identifiers, numeric measures, and categorical attributes. The main preprocessing requirement is converting date-related columns from string to datetime, which will be handled during feature engineering and downstream analysis